# MBIR Magnetic-Field Reconstruction: Real-Data Workflow

This notebook walks through a complete **real-data workflow** for recovering the
in-plane magnetic field from an electron holography phase image, using
`libertem_holo.base.mbir` with [`unxt`](https://github.com/GalileoAstronomia/unxt)
for explicit physical-unit management.

## Workflow steps
1. Load a pre-reconstructed magnetic phase map
2. Specify microscope parameters as `unxt.Quantity` objects
3. Choose the regularization parameter via the L-curve
4. Reconstruct the in-plane B-field with Tikhonov regularization
5. Optionally apply the pyramid method for a multi-scale result
6. Inspect and visualise the output fields (with units)

> **Note**: This notebook uses a synthetic phase image that mimics a
> real experiment. Replace the synthetic phase image in *Section 2* with your
> own loaded phase map to process real data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import unxt

from libertem_holo.base.mbir import (
    b_field_to_phase,
    compute_lcurve,
    phase_scale_factor,
    phase_to_b_field,
    plot_b_field,
    plot_lcurve,
    reconstruct_b_field_pyramid,
    reconstruct_b_field_tikhonov,
)

## 1 · Microscope and sample parameters

Set your experimental parameters here as `unxt.Quantity` objects.

| Parameter | Typical range | Notes |
|-----------|--------------|-------|
| `voxel_size` | 1–20 nm | Real-space pixel size of the reconstructed phase |
| `thickness` | 10–200 nm | Projected sample thickness (from EELS or model) |

Using explicit units avoids common mistakes such as mixing nm and m.

In [ ]:
# ── Edit these values to match your experiment ──────────────────────────────
voxel_size = unxt.Quantity(4.0, 'nm')   # pixel size of the phase image
thickness  = unxt.Quantity(60.0, 'nm')  # projected sample thickness
# ────────────────────────────────────────────────────────────────────────────

C = phase_scale_factor(voxel_size=voxel_size, thickness=thickness)
print(f"voxel_size : {voxel_size}")
print(f"thickness  : {thickness}")
print(f"Phase scale factor C = {C}")
print(f"  → a phase gradient of 1 rad/pixel corresponds to "
      f"{1/float(C.value)*1000:.2f} mT")

## 2 · Load phase image

Replace the synthetic data block below with code that loads your real
magnetic phase image. The phase should:
- Be a 2-D NumPy array in **radians**
- Have the electrostatic phase contribution **subtracted** (e.g. from a
  reference hologram or by the flipping series method)
- Be wrapped in a `unxt.Quantity` with unit `'rad'`

In [ ]:
# ─── Replace with your data loading code ─────────────────────────────────────
# Example:
#   import tifffile
#   phase_arr = tifffile.imread('magnetic_phase.tiff')
#   phase = unxt.Quantity(phase_arr, 'rad')

# Synthetic substitute: magnetic stripe domain pattern
rng = np.random.default_rng(7)

ny, nx = 256, 256
y_coords, x_coords = np.mgrid[-ny//2:ny//2, -nx//2:nx//2]

# Stripe domains alternating in x-direction
period = 32   # pixels
az_stripe = np.sin(2 * np.pi * x_coords / period) * np.exp(
    -(y_coords**2) / (2 * (ny / 6)**2)
)

dy_Az, dx_Az = np.gradient(az_stripe)
bx_known = dy_Az
by_known = -dx_Az

# Forward model: ideal (noise-free) magnetic phase
phase_ideal = b_field_to_phase(
    unxt.Quantity(bx_known, 'T'),
    unxt.Quantity(by_known, 'T'),
    voxel_size=voxel_size,
    thickness=thickness,
)

# Add realistic noise level
noise_sigma = 0.03  # rad
phase_arr = np.asarray(phase_ideal.value) + rng.standard_normal((ny, nx)) * noise_sigma

# ─── End of synthetic data block ─────────────────────────────────────────────

# Wrap as a unxt.Quantity – this is the required input format
phase = unxt.Quantity(phase_arr, 'rad')

print(f"Phase shape : {phase_arr.shape}")
print(f"Phase unit  : {phase.unit}")
print(f"Phase range : [{phase_arr.min():.3f}, {phase_arr.max():.3f}] {phase.unit}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(phase_arr, cmap='RdBu_r', origin='lower')
ax.set_title(f'Magnetic phase [{phase.unit}]')
pix_nm = float(unxt.ustrip('nm', voxel_size))
ax.set_xlabel(f'x [pixel × {pix_nm:.1f} nm]')
ax.set_ylabel(f'y [pixel × {pix_nm:.1f} nm]')
plt.colorbar(im, ax=ax, label=str(phase.unit))
plt.tight_layout()
plt.show()

## 3 · L-curve: choose regularization parameter λ

The L-curve method tests a range of regularization parameters and identifies
the *corner* (maximum curvature) as the optimal trade-off between data
fidelity and solution smoothness.

In [ ]:
lcurve = compute_lcurve(
    phase,
    voxel_size=voxel_size,
    thickness=thickness,
    n_lambdas=50,
    lambda_min=1e-8,
    lambda_max=1e6,
)

lam_opt = lcurve.optimal_lambda
print(f"Optimal λ (L-curve corner): {lam_opt:.2e}")

In [ ]:
plot_lcurve(lcurve)
plt.title('L-curve: residual norm vs solution norm')
plt.show()

### Optional: override λ manually

If the automatic L-curve selection is not satisfactory, override the value:

In [ ]:
# Uncomment and set a manual value if needed:
# lam_opt = 1e-2
print(f"Using λ = {lam_opt:.2e}")

## 4 · Tikhonov MBIR reconstruction

Reconstruct the in-plane magnetic field components $B_x$ and $B_y$.
The output is an `MBIRResult` whose fields are `unxt.Quantity` objects with
explicit magnetic-flux-density units.

In [ ]:
result = reconstruct_b_field_tikhonov(
    phase,
    voxel_size=voxel_size,
    thickness=thickness,
    regularization_parameter=lam_opt,
    output_unit='mT',   # choose 'T' or 'mT'
)

# All outputs are unxt.Quantity objects
print(f"B_x  unit : {result.b_x.unit}")
print(f"B_y  unit : {result.b_y.unit}")
print(f"|B|  unit : {result.b_magnitude.unit}")
print(f"Voxel size: {result.voxel_size}")
print(f"λ used    : {result.regularization_parameter:.2e}")

bx_arr = np.asarray(result.b_x.value)
by_arr = np.asarray(result.b_y.value)
bmag_arr = np.asarray(result.b_magnitude.value)
print(f"|B| max   : {bmag_arr.max():.2f} {result.b_magnitude.unit}")
print(f"|B| mean  : {bmag_arr.mean():.2f} {result.b_magnitude.unit}")

In [ ]:
plot_b_field(result)
plt.suptitle(
    f'Tikhonov MBIR  (λ = {lam_opt:.2e}, '
    f'thickness = {thickness}, voxel = {voxel_size})',
    y=1.02
)
plt.tight_layout()
plt.show()

## 5 · Pyramid (multi-scale) reconstruction

The pyramid method decomposes the reconstruction across multiple spatial scales.
Increase `n_levels` for smoother results at the cost of slightly more computation.

In [ ]:
result_pyr = reconstruct_b_field_pyramid(
    phase,
    voxel_size=voxel_size,
    thickness=thickness,
    regularization_parameter=lam_opt,
    n_levels=3,
    output_unit='mT',
)

print(f"Pyramid |B| unit: {result_pyr.b_magnitude.unit}")
print(f"Pyramid |B| max : {np.asarray(result_pyr.b_magnitude.value).max():.2f} {result_pyr.b_magnitude.unit}")

In [ ]:
plot_b_field(result_pyr)
plt.suptitle(
    f'Pyramid MBIR  (λ = {lam_opt:.2e}, 3 levels, '
    f'thickness = {thickness})',
    y=1.02
)
plt.tight_layout()
plt.show()

## 6 · Forward-model verification

Verify the reconstruction by applying the forward model to the recovered field
and comparing with the input phase.

In [ ]:
# Convert result back to T for the forward model
bx_T = unxt.uconvert('T', result.b_x)
by_T = unxt.uconvert('T', result.b_y)

phase_reconstructed = b_field_to_phase(
    bx_T,
    by_T,
    voxel_size=voxel_size,
    thickness=thickness,
)

residual = np.asarray(phase_reconstructed.value) - phase_arr
rel_residual = np.linalg.norm(residual) / np.linalg.norm(phase_arr)
print(f"Forward-model residual ||Aφ_rec − φ|| / ||φ|| = {rel_residual:.4f}")
print(f"(smaller is better; expected ~λ/(C²+λ) for large λ)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(phase_arr, cmap='RdBu_r', origin='lower')
axes[0].set_title(f'Input phase [{phase.unit}]')
plt.colorbar(axes[0].images[0], ax=axes[0])

axes[1].imshow(np.asarray(phase_reconstructed.value), cmap='RdBu_r', origin='lower')
axes[1].set_title(f'Reconstructed phase [{phase_reconstructed.unit}]')
plt.colorbar(axes[1].images[0], ax=axes[1])

axes[2].imshow(residual, cmap='RdBu_r', origin='lower')
axes[2].set_title('Residual phase [rad]')
plt.colorbar(axes[2].images[0], ax=axes[2])

pix_nm = float(unxt.ustrip('nm', voxel_size))
for ax in axes:
    ax.set_xlabel(f'x [pixel × {pix_nm:.1f} nm]')
    ax.set_ylabel(f'y [pixel × {pix_nm:.1f} nm]')

plt.tight_layout()
plt.show()

## 7 · Export results

Save the reconstructed B-field arrays. The units are recorded in the metadata.

In [ ]:
# Example: save as NumPy .npz with unit metadata
# np.savez(
#     'b_field_reconstruction.npz',
#     b_x=np.asarray(result.b_x.value),
#     b_y=np.asarray(result.b_y.value),
#     b_magnitude=np.asarray(result.b_magnitude.value),
#     unit=str(result.b_x.unit),
#     voxel_size_nm=float(unxt.ustrip('nm', result.voxel_size)),
#     regularization_parameter=result.regularization_parameter,
# )

print("Results summary:")
print(f"  B_x  : shape={np.asarray(result.b_x.value).shape}, unit={result.b_x.unit}")
print(f"  B_y  : shape={np.asarray(result.b_y.value).shape}, unit={result.b_y.unit}")
print(f"  |B|  : shape={np.asarray(result.b_magnitude.value).shape}, unit={result.b_magnitude.unit}")
print(f"  voxel: {result.voxel_size}")
print(f"  λ    : {result.regularization_parameter:.2e}")